# 4. 节点中访问状态

## 4.1 图节点读取状态

在 LangGraph 中，节点本质上是一个可调用对象，通常定义为普通 Python 函数。节点函数被执行时，LangGraph 会自动将当前图运行到该节点的 `State` 传入节点函数。

节点函数的第一个参数通常是当前运行图的状态对象，也就是 `State`。

```Python
def node(state: StateSchema):
    ...
```
其中，`state` 表示当前节点执行时可以访问到的全局状态快照。节点可以通过读取 `state` 中的字段获取上有节点写入的数据，并给予这些数据完成当前节点的业务逻辑。

In [4]:
from langgraph.graph import START, END, StateGraph
from typing import TypedDict, Annotated
from operator import add

class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) ->  OverAllState:
   for k, v in state.items():
       print(f"k: {k}, v: {v}")

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

graph = builder.compile()
result = graph.invoke({"logs": ["START"], "cur_id": "start"})

k: logs, v: ['START']
k: cur_id, v: start


## 4.2 图节点更新状态

在 LangGraph 中，节点函数通常不需要返回更新后的完整状态，只需要返回本节点对状态的局部更新。

也就是说，节点的返回值可以只包含需要修改的状态字段。

- 对于节点没有返回的字段，LangGraph 会保留其原有状态值；
- 对于节点返回的字段，LangGraph 会根据该字段是否配置了 Reducer 来决定如何合并更新值。
    - 如果字段配置了 Reducer，则使用对应的 Reducer 函数将旧值和新值合并；
    - 如果字段没有配置 Reducer，则按照默认规则使用节点返回的新值覆盖原值。

LangGraph运行时会按照状态字段的Reducer函数将其与当前的最新状态合并。

In [5]:
from langgraph.graph import START, END, StateGraph
from typing import TypedDict, Annotated
from operator import add

class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) ->  OverAllState:
   for k, v in state.items():
       print(f"k: {k}, v: {v}")
   return {
       "logs": ["node_1 更新状态"]
   }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

graph = builder.compile()
result = graph.invoke({"logs": ["START"], "cur_id": "start"})
print("=" * 30, "-> result <-", "=" * 30)
print(result)

k: logs, v: ['START']
k: cur_id, v: start
============================== -> result <- ==============================
{'logs': ['START', 'node_1 更新状态'], 'cur_id': 'start'}


在上述示例中：
- `logs` 字段通过 `Annotated[list[str], add]` 绑定了 `Reducer` 函数 `operator.add`，因此 LangGraph 会将原有的 `logs` 值和 `node_1` 返回新 `logs` 值进行列表拼接；
- `cur+id` 字段没有出现在 `node_1` 的返回值中，因此该字段不会被更新。图运行结束后，输出状态的 `cur_id` 仍然爆出输入的值。

## 4.3. Overwrite绕过Reducer
在前面的示例中，如果某个状态字段定义了 `Reducer`，那么节点返回该字段的更新值时，LangGraph 默认会通过对应的 `Reducer` 将新值与已有状态值进行合并。

在某些场景下，我们可能并不希望继续执行 `Reducer` 的聚合逻辑，而是希望本次更新直接覆盖旧值。这时可以使用 `Overwrite`。

`Overwrite` 的作用是：告诉 LangGraph 本次状态更新不走该字段原本定义的 `Reducer`，而是直接用新值覆盖状态中的旧值。

需要注意的是，`Overwrite` 只影响当前这一次更新，并不会修改状态字段本身的 `Reducer` 定义。后续节点如果继续正常返回该字段的更新值，仍然会按照原来的 `Reducer` 逻辑进行合并。

In [8]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Overwrite
from typing import TypedDict, Annotated
from operator import add

class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState):
    return {
        "logs": ["node_1"],
        "cur_id": "node_1"
    }

def node_2(state: OverAllState):
    return {
        # Overwrite
        "logs": Overwrite(["node_2"]),
        "cur_id": "node_2"
    }

def node_3(state: OverAllState):
    return {
        "logs": ["node_3"],
        "cur_id": "node_3"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
result = graph.invoke({"logs": ["START"], "cur_id": "start"})
print('=' * 30, '-> result <-', '=' * 30)
print(result)

============================== -> result <- ==============================
{'logs': ['node_2', 'node_3'], 'cur_id': 'node_3'}


In [12]:
from time import sleep
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add

class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: Annotated[str, add]

def node_1(state: OverAllState):
    for k, v in state.items():
        print(f"== 1 == k: {k}, v: {v}")
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": "node_1 "
    }

def node_2(state: OverAllState):
    for k, v in state.items():
        print(f"== 2 == k: {k}, v: {v}")
    return {
        "logs": ["node_2 运行完毕"],
        "cur_id": "node_2 "
    }

def node_3(state: OverAllState):
    sleep(1)
    for k, v in state.items():
        print(f"== 3 == k: {k}, v: {v}")
    return {
        "logs": ["node_3 运行完毕"],
        "cur_id": "node_3 "
    }

def node_4(state: OverAllState):
    sleep(2)
    for k, v in state.items():
        print(f"== 4 == k: {k}, v: {v}")
    return {
        "logs": ["node_4 运行完毕"],
        "cur_id": "node_4 "
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_node("node_4", node_4)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_1", "node_3")
builder.add_edge("node_2", "node_4")
builder.add_edge("node_3", "node_4")
builder.add_edge("node_4", END)

graph = builder.compile()
result = graph.invoke({"logs": ["START"], "cur_id": "start "})
print('=' * 30, '-> result <-', '=' * 30)
print(result)

== 1 == k: logs, v: ['START']
== 1 == k: cur_id, v: start 
== 2 == k: logs, v: ['START', 'node_1 运行完毕']
== 2 == k: cur_id, v: start node_1 
== 3 == k: logs, v: ['START', 'node_1 运行完毕']
== 3 == k: cur_id, v: start node_1 
== 4 == k: logs, v: ['START', 'node_1 运行完毕', 'node_2 运行完毕', 'node_3 运行完毕']
== 4 == k: cur_id, v: start node_1 node_2 node_3 
============================== -> result <- ==============================
{'logs': ['START', 'node_1 运行完毕', 'node_2 运行完毕', 'node_3 运行完毕', 'node_4 运行完毕'], 'cur_id': 'start node_1 node_2 node_3 node_4 '}


如果出现并行节点，同时更新状态，往下游节点传递的时候，必须要有 `Reducer`